# Notebook 4 - Walk-Forward Backtesting
Expanding-window time-series backtest for return, volatility and loss prediction.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBRegressor, XGBClassifier

df = pd.read_csv("data/processed/ml_features.csv", index_col=0, parse_dates=True)

target_return="Target_Return"
target_vol="Target_Volatility"
target_loss="Target_Loss"

features=[c for c in df.columns if c not in [target_return,target_vol,target_loss]]


## Walk-forward settings

In [ ]:
INITIAL_TRAIN=756      # ~3 years
STEP=21                # retrain every month
HORIZON=21             # predict next month

results=[]

In [ ]:
for start in range(INITIAL_TRAIN, len(df)-HORIZON, STEP):

    train=df.iloc[:start]
    test=df.iloc[start:start+HORIZON]

    X_train=train[features]
    X_test=test[features]

    scaler=StandardScaler()
    X_train_sc=scaler.fit_transform(X_train)
    X_test_sc=scaler.transform(X_test)

    # ---------------- Return ----------------

    y_train=train[target_return]
    y_test=test[target_return]

    lr=LinearRegression()
    lr.fit(X_train_sc,y_train)
    pred_lr=lr.predict(X_test_sc)

    rf=RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train,y_train)
    pred_rf=rf.predict(X_test)

    xgb=XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        objective="reg:squarederror",
        random_state=42
    )
    xgb.fit(X_train,y_train)
    pred_xgb=xgb.predict(X_test)

    # ---------------- Volatility ----------------

    yv_train=train[target_vol]
    yv_test=test[target_vol]

    rf_vol=RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )
    rf_vol.fit(X_train,yv_train)
    pred_vol=rf_vol.predict(X_test)

    # ---------------- Classification ----------------

    yc_train=train[target_loss]
    yc_test=test[target_loss]

    clf=RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    )
    clf.fit(X_train,yc_train)

    pred_loss=clf.predict(X_test)
    prob_loss=clf.predict_proba(X_test)[:,1]

    for i,date in enumerate(test.index):

        results.append({
            "Date":date,
            "Actual_Return":y_test.iloc[i],
            "LR_Return":pred_lr[i],
            "RF_Return":pred_rf[i],
            "XGB_Return":pred_xgb[i],
            "Actual_Volatility":yv_test.iloc[i],
            "Pred_Volatility":pred_vol[i],
            "Actual_Loss":yc_test.iloc[i],
            "Pred_Loss":pred_loss[i],
            "Loss_Probability":prob_loss[i]
        })


In [ ]:
pred=pd.DataFrame(results).set_index("Date")

def reg_metrics(actual,prediction):
    return {
        "MAE":mean_absolute_error(actual,prediction),
        "RMSE":np.sqrt(mean_squared_error(actual,prediction)),
        "R2":r2_score(actual,prediction),
        "Direction":(np.sign(actual)==np.sign(prediction)).mean()
    }

return_metrics=pd.DataFrame({
    "Linear":reg_metrics(pred.Actual_Return,pred.LR_Return),
    "RandomForest":reg_metrics(pred.Actual_Return,pred.RF_Return),
    "XGBoost":reg_metrics(pred.Actual_Return,pred.XGB_Return)
}).T

loss_metrics=pd.DataFrame([{
    "Accuracy":accuracy_score(pred.Actual_Loss,pred.Pred_Loss),
    "Precision":precision_score(pred.Actual_Loss,pred.Pred_Loss),
    "Recall":recall_score(pred.Actual_Loss,pred.Pred_Loss),
    "F1":f1_score(pred.Actual_Loss,pred.Pred_Loss),
    "ROC_AUC":roc_auc_score(pred.Actual_Loss,pred.Loss_Probability)
}])

print(return_metrics)
print(loss_metrics)


In [ ]:
pred.to_csv("output/walkforward_predictions.csv")
return_metrics.to_csv("output/return_metrics.csv")
loss_metrics.to_csv("output/loss_metrics.csv")

pred.head()


## Power BI

Use **walkforward_predictions.csv** as the primary ML table.

The next notebook will add:
- Efficient Frontier
- Maximum Sharpe portfolio
- Minimum Variance portfolio
- Monte Carlo simulation
- Portfolio optimization
